Purpose: Filter genes for input to XGBoost.<br>
Author: Anna Pardo<br>
Date initiated: Feb. 17, 2026

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold

In [2]:
# load input data
indata = pd.read_csv("./paired_TPM_physiology.txt",sep="\t",header="infer")
indata.head()

/tmp/ipykernel_19819/404855715.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  indata = pd.read_csv("./paired_TPM_physiology.txt",sep="\t",header="infer")


,sample_name,genotype,treat,ZT,photo,cond,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g
0,Y111,18,D,1.0,2.627596,0.026132,48.380647,6.981615,0.00000,19.211534,...,0.000000,5.500249,2.171056,0.671994,8.838875,0.372084,2.805992,4.407968,0.0,0.000000
1,Y123,18,D,1.0,2.679172,0.021603,52.585871,5.331016,0.57518,12.811805,...,0.101045,11.410506,2.456701,0.464693,10.125277,0.327475,1.058391,4.115716,0.0,0.000000
2,Y120,18,D,3.0,1.211527,0.007856,41.192373,5.821853,0.00000,16.055452,...,0.000000,5.037032,2.061853,0.896509,10.437337,0.302887,1.945765,5.470556,0.0,0.212331
3,Y125,18,D,3.0,1.289725,0.009783,49.458436,6.239587,0.00000,20.374043,...,0.000000,6.796193,3.347733,0.445386,9.763749,0.422761,2.479682,5.295689,0.0,0.000000
4,Y129,18,D,3.0,1.549238,0.010471,45.312890,7.263022,0.00000,19.973103,...,0.116594,7.899789,1.574850,0.584944,9.166182,0.539808,1.944959,7.736828,0.0,0.000000


In [5]:
# filter to only drought samples
droughtonly = indata[indata["treat"]=="D"]
len(droughtonly.index)

211

In [6]:
len(droughtonly.columns)-6

85962

Given that I have many more genes (features) than observations, the curse of dimensionality is very much an issue here. Some non-arbitrary gene sets I have as options to take care of that:<br>
- maSigPro clusters peaking at different ZTs (for each genotype...would need to handle this diversity somehow)
- HEB genes (biased to Yf or biased to Ya)
- recently generated: expression partitions from HybridExpress (again, for each genotype)
- CAM genes

Also, I should remove zero-variance features.

In [8]:
# function for filtering out zero-variance features
# define a function from an answer in https://stackoverflow.com/questions/39812885/retain-feature-names-after-scikit-feature-selection
def variance_threshold_selector(data):
    selector = VarianceThreshold()
    selector.fit(data)
    return data[data.columns[selector.get_support(indices=True)]]